# Data Visualization: International Education Costs Dataset

This notebook contains comprehensive visualizations of the International Education Costs dataset using matplotlib and seaborn. It includes univariate, bivariate, and multivariate analyses.

In [ ]:
"""
    Setup and Data Loading
    
    Initialize visualization libraries and load the dataset.
    """
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configure visualization settings
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Load and prepare data
df = pd.read_csv('International_Education_Costs.csv')
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumns:", df.columns.tolist())

## Section 1: Univariate Analysis

In [ ]:
"""
    Univariate Plot 1: Histogram with KDE Overlay
    
    Shows the frequency distribution of a continuous variable
    with a kernel density estimate overlay.
    """
    
fig = plt.figure(figsize=(15, 10))

numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

for idx, col in enumerate(numerical_cols, 1):
    ax = plt.subplot(2, 3, idx)
    
    # Create histogram with KDE
    sns.histplot(data=df, x=col, kde=True, stat='density', bins=25, 
                 color='steelblue', edgecolor='black', alpha=0.7, ax=ax)
    
    ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('01_histograms_kde.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Histograms with KDE overlays created")

In [ ]:
"""
    Univariate Plot 2: Boxplots with Outlier Detection
    
    Visualizes distribution, quartiles, and outliers for numerical variables.
    """
    
fig = plt.figure(figsize=(15, 8))

for idx, col in enumerate(numerical_cols, 1):
    ax = plt.subplot(2, 3, idx)
    
    # Create boxplot
    bp = ax.boxplot([df[col]], vert=True, patch_artist=True,
                     widths=0.5,
                     boxprops=dict(facecolor='lightblue', alpha=0.7, linewidth=1.5),
                     whiskerprops=dict(color='black', linewidth=1.5),
                     capprops=dict(color='black', linewidth=1.5),
                     medianprops=dict(color='red', linewidth=2.5),
                     flierprops=dict(marker='o', markerfacecolor='red', markersize=8, alpha=0.6))
    
    ax.set_ylabel(col, fontsize=10)
    ax.set_title(f'Boxplot: {col}', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add count of outliers
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]
    ax.text(1.15, ax.get_ylim()[1], f'Outliers: {len(outliers)}', 
            fontsize=9, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.savefig('02_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Boxplots with outlier information created")

In [ ]:
"""
    Univariate Plot 3: Count Plots for Categorical Variables
    
    Shows the frequency of categories in categorical variables.
    """
    
categorical_cols = df.select_dtypes(include=['object']).columns

fig = plt.figure(figsize=(16, 6))

for idx, col in enumerate(categorical_cols[:3], 1):  # Top 3 categorical columns
    ax = plt.subplot(1, 3, idx)
    
    # Count plot
    top_categories = df[col].value_counts().head(8)
    sns.barplot(x=top_categories.values, y=top_categories.index, palette='Set2', ax=ax)
    
    ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Count', fontsize=10)
    ax.set_ylabel(col, fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(top_categories.values):
        ax.text(v + 0.1, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('03_categorical_counts.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Categorical count plots created")

In [ ]:
"""
    Univariate Plot 4: Violin Plots
    
    Combines boxplot and KDE to show distribution shape for each variable.
    """
    
fig = plt.figure(figsize=(15, 8))

for idx, col in enumerate(numerical_cols, 1):
    ax = plt.subplot(2, 3, idx)
    
    # Violin plot
    parts = ax.violinplot([df[col]], positions=[0], showmeans=True, showmedians=True)
    
    # Customize colors
    for pc in parts['bodies']:
        pc.set_facecolor('lightblue')
        pc.set_alpha(0.7)
    
    ax.set_ylabel(col, fontsize=10)
    ax.set_title(f'Violin Plot: {col}', fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('04_violin_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Violin plots created")

## Section 2: Bivariate Analysis

In [ ]:
"""
    Bivariate Plot 1: Scatter Plots with Trend Lines
    
    Visualizes relationships between pairs of numerical variables.
    """
    
numerical_data = df.select_dtypes(include=['float64', 'int64'])
if len(numerical_data.columns) >= 2:
    col1, col2 = numerical_data.columns[0], numerical_data.columns[1]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Scatter plot with regression line
    sns.regplot(x=col1, y=col2, data=df, scatter_kws={'alpha': 0.5, 's': 80, 'color': 'steelblue'},
                line_kws={'color': 'red', 'linewidth': 2}, ax=ax)
    
    ax.set_title(f'Scatter Plot: {col1} vs {col2} with Trend Line', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel(col1, fontsize=12)
    ax.set_ylabel(col2, fontsize=12)
    ax.grid(alpha=0.3)
    
    # Calculate and display correlation
    corr = df[col1].corr(df[col2])
    ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax.transAxes,
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig('05_scatter_with_trend.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Scatter plot created (Correlation: {corr:.3f})")

In [ ]:
"""
    Bivariate Plot 2: Correlation Heatmap
    
    Visualizes all pairwise correlations between numerical variables.
    """
    
corr_matrix = df.select_dtypes(include=['float64', 'int64']).corr()

fig, ax = plt.subplots(figsize=(10, 8))

# Create heatmap with annotations
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=2, cbar_kws={'shrink': 0.8},
            vmin=-1, vmax=1, ax=ax)

ax.set_title('Correlation Heatmap: Numerical Variables', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('06_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation heatmap created")

In [ ]:
"""
    Bivariate Plot 3: Joint Distribution Plots
    
    Shows both marginal and joint distributions for two variables.
    """
    
if len(numerical_data.columns) >= 2:
    col1, col2 = numerical_data.columns[0], numerical_data.columns[1]
    
    # Create figure with subplots for joint plot
    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Main scatter plot
    ax_main = fig.add_subplot(gs[1:, :-1])
    ax_main.scatter(df[col1], df[col2], alpha=0.5, s=50, color='steelblue')
    ax_main.set_xlabel(col1, fontsize=11)
    ax_main.set_ylabel(col2, fontsize=11)
    ax_main.set_title(f'Joint Distribution: {col1} vs {col2}', fontsize=12, fontweight='bold')
    ax_main.grid(alpha=0.3)
    
    # Top histogram
    ax_top = fig.add_subplot(gs[0, :-1], sharex=ax_main)
    sns.histplot(data=df, x=col1, kde=True, ax=ax_top, color='steelblue', legend=False)
    ax_top.set_ylabel('Frequency')
    ax_top.set_title(f'Distribution of {col1}')
    
    # Right histogram
    ax_right = fig.add_subplot(gs[1:, -1], sharey=ax_main)
    sns.histplot(data=df, y=col2, kde=True, ax=ax_right, color='coral', legend=False)
    ax_right.set_xlabel('Frequency')
    ax_right.set_title(f'Distribution of {col2}')
    
    plt.savefig('07_joint_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Joint distribution plot created")

In [ ]:
"""
    Bivariate Plot 4: Box Plots by Categories
    
    Shows numerical variable distributions across categorical groups.
    """
    
if len(categorical_cols) > 0 and len(numerical_cols) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Select columns with reasonable number of categories
    cat_col = categorical_cols[0]
    num_col = numerical_cols[0]
    
    # Filter to top categories for clarity
    top_cats = df[cat_col].value_counts().head(10).index
    df_filtered = df[df[cat_col].isin(top_cats)]
    
    # Create boxplot
    sns.boxplot(data=df_filtered, x=cat_col, y=num_col, palette='Set2', ax=ax)
    
    ax.set_title(f'{num_col} Distribution by {cat_col}', fontsize=14, fontweight='bold')
    ax.set_xlabel(cat_col, fontsize=12)
    ax.set_ylabel(num_col, fontsize=12)
    plt.xticks(rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('08_boxplot_by_category.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Categorical boxplot created")

## Section 3: Multivariate Analysis

In [ ]:
"""
    Multivariate Plot 1: Pairplot
    
    Creates a grid of scatter and histogram plots showing all pairwise relationships.
    """
    
# Create pairplot with selected numerical columns
if len(numerical_cols) <= 5:
    pairplot = sns.pairplot(df[numerical_cols], diag_kind='kde', 
                             plot_kws={'alpha': 0.6, 's': 80},
                             diag_kws={'shade': True})
    
    pairplot.fig.suptitle('Pairplot: All Numerical Variables', 
                          fontsize=14, fontweight='bold', y=1.001)
    
    plt.savefig('09_pairplot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Pairplot created")
else:
    print("Dataset has too many numerical variables for pairplot. Showing subset instead.")
    pairplot = sns.pairplot(df[numerical_cols[:4]], diag_kind='kde',
                             plot_kws={'alpha': 0.6, 's': 80})
    plt.savefig('09_pairplot_subset.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
"""
    Multivariate Plot 2: Heatmap with Categorical Data Encoded
    
    Shows correlations and relationships including encoded categorical variables.
    """
    
# Prepare data with encoded categorical variables
df_encoded = df.copy()
for col in categorical_cols:
    df_encoded[col + '_encoded'] = pd.factorize(df_encoded[col])[0]

# Select numerical columns for correlation
cols_for_corr = list(numerical_cols) + [col + '_encoded' for col in categorical_cols[:2]]
if len(cols_for_corr) <= 10:
    corr_matrix_extended = df_encoded[cols_for_corr].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix_extended, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1.5, cbar_kws={'shrink': 0.8},
                vmin=-1, vmax=1, ax=ax)
    
    ax.set_title('Extended Correlation Heatmap (Including Encoded Variables)', 
                 fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('10_extended_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Extended correlation heatmap created")

In [ ]:
"""
    Multivariate Plot 3: Subgroup Analysis
    
    Compares distributions across different categorical subgroups.
    """
    
if len(categorical_cols) > 0 and len(numerical_cols) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    cat_col = categorical_cols[0]
    num_col = numerical_cols[0]
    
    top_cats = df[cat_col].value_counts().head(5).index
    df_subset = df[df[cat_col].isin(top_cats)]
    
    # Subplot 1: Distribution by category
    for cat in top_cats:
        data = df_subset[df_subset[cat_col] == cat][num_col]
        axes[0].hist(data, alpha=0.5, label=str(cat), bins=20)
    
    axes[0].set_title(f'Distribution of {num_col} by {cat_col}', 
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel(num_col)
    axes[0].set_ylabel('Frequency')
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Subplot 2: KDE by category
    for cat in top_cats:
        data = df_subset[df_subset[cat_col] == cat][num_col]
        data.plot.kde(ax=axes[1], label=str(cat), linewidth=2)
    
    axes[1].set_title(f'KDE Distribution of {num_col} by {cat_col}', 
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel(num_col)
    axes[1].set_ylabel('Density')
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('11_subgroup_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Subgroup analysis created")

## Section 4: Advanced Visualizations

In [ ]:
"""
    Advanced Plot 1: Distribution Comparison (Q-Q Plot)
    
    Compares each numerical variable against a normal distribution.
    """
    
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    ax = axes[idx]
    stats.probplot(df[col].dropna(), dist="norm", plot=ax)
    ax.set_title(f'Q-Q Plot: {col}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)

if len(numerical_cols) < 6:
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('12_qq_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Q-Q plots created")

In [ ]:
"""
    Advanced Plot 2: FacetGrid with Multiple Dimensions
    
    Creates separate plots for different subgroups of data.
    """
    
if len(categorical_cols) > 0 and len(numerical_cols) > 1:
    cat_col = categorical_cols[0]
    num_col = numerical_cols[0]
    
    # Get top categories
    top_cats = df[cat_col].value_counts().head(4).index
    df_subset = df[df[cat_col].isin(top_cats)]
    
    # Create FacetGrid
    g = sns.FacetGrid(df_subset, col=cat_col, col_wrap=2, height=5, aspect=1.2)
    g.map(sns.histplot, num_col, kde=True, bins=20, color='steelblue')
    g.set_titles(f'Distribution of {{col_name}}')
    
    plt.savefig('13_facet_grid.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ FacetGrid visualization created")

In [ ]:
"""
    Advanced Plot 3: Outlier Visualization
    
    Highlights outliers using different techniques.
    """
    
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    ax = axes[idx]
    
    # Calculate outliers using IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Separate outliers and inliers
    inliers = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    # Plot
    ax.scatter(range(len(inliers)), inliers[col], alpha=0.5, s=30, 
               color='steelblue', label=f'Inliers (n={len(inliers)})')
    ax.scatter(range(len(inliers), len(inliers) + len(outliers)), outliers[col], 
               alpha=0.8, s=100, color='red', marker='X', label=f'Outliers (n={len(outliers)})')
    
    ax.axhline(y=upper_bound, color='orange', linestyle='--', linewidth=2, label='Upper Bound')
    ax.axhline(y=lower_bound, color='orange', linestyle='--', linewidth=2, label='Lower Bound')
    
    ax.set_title(f'Outlier Detection: {col}', fontsize=11, fontweight='bold')
    ax.set_ylabel(col, fontsize=10)
    ax.set_xlabel('Index', fontsize=10)
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.3)

if len(numerical_cols) < 6:
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('14_outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Outlier detection visualization created")

## Summary

In [ ]:
"""
    Visualization Summary
    
    Lists all visualizations created.
    """
    
print("\n" + "="*70)
print(" " * 15 + "VISUALIZATION SUMMARY")
print("="*70)

visualizations = [
    ("01_histograms_kde.png", "Histograms with KDE Overlays"),
    ("02_boxplots.png", "Boxplots with Outlier Detection"),
    ("03_categorical_counts.png", "Categorical Variable Count Plots"),
    ("04_violin_plots.png", "Violin Plots"),
    ("05_scatter_with_trend.png", "Scatter Plots with Trend Lines"),
    ("06_correlation_heatmap.png", "Correlation Heatmap"),
    ("07_joint_distribution.png", "Joint Distribution Plots"),
    ("08_boxplot_by_category.png", "Boxplots by Categorical Groups"),
    ("09_pairplot.png", "Pairplot: All Variables"),
    ("10_extended_heatmap.png", "Extended Correlation with Encoded Variables"),
    ("11_subgroup_analysis.png", "Subgroup Distribution Analysis"),
    ("12_qq_plots.png", "Q-Q Plots (Normality Check)"),
    ("13_facet_grid.png", "FacetGrid: Multi-dimensional Analysis"),
    ("14_outlier_detection.png", "Outlier Detection Visualizations")
]

print("\nUnivariate Analysis:")
for fname, desc in visualizations[:4]:
    print(f"  • {desc}")

print("\nBivariate Analysis:")
for fname, desc in visualizations[4:8]:
    print(f"  • {desc}")

print("\nMultivariate Analysis:")
for fname, desc in visualizations[8:11]:
    print(f"  • {desc}")

print("\nAdvanced Visualizations:")
for fname, desc in visualizations[11:]:
    print(f"  • {desc}")

print("\n" + "="*70)
print(f"Total Visualizations: {len(visualizations)}")
print(f"Images saved with 300 DPI for publication quality")
print("="*70 + "\n")